# R8 · Probabilités de base — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r8_probabilites/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r8_probabilites/03_notebook.ipynb`, créée par `python tools/start_chapter.py R8`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R8.1 à R8.10, R8.13 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | R8.16 | Factorielle et combinaisons · `factorial`, `binomial` | 🔨 | ★ | 15 |
|  | R8.17 | Une probabilité exacte par énumération · `enumerate_probability` | 🔨 | ★★ | 25 |
|  | R8.18 | Espérance et variance d'une loi · `expectation`, `variance` | 🔨 | ★ | 15 |
|  | R8.19 | Estimer une probabilité par simulation · `simulate_frequency` | 🔨 | ★★ | 20 |
| B | R8.20 | Prédire : dans une classe de 30, deux élèves ont-ils le même anniversaire ? | 🔮 | ★★ | 15 |
|  | R8.21 | La loi des grands nombres : fréquence et nombre d'essais | 🔬 | ★★ | 20 |
|  | R8.22 | Le diagramme en bâtons d'une loi | 🎨 | ★ | 10 |
| C | R8.23 | Chasse à l'erreur : trois raisonnements de probabilités faux | 🐛 | ★★ | 20 |
|  | R8.24 | Reproductibilité : générateur aléatoire et graine | 🛠️ | ★ | 15 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R8")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : univers et événements, probabilités, dénombrement, arbres, indépendance, lois, espérance et variance.
- Écrire les six fonctions de `mymath/proba_basics.py` et les tester.
- Prédire une probabilité qui surprend, mesurer la loi des grands nombres, tracer une loi, chasser des raisonnements faux et rendre une expérience reproductible.

**Rappel express.** Équiprobabilité : $P(A) = \frac{|A|}{|\Omega|}$ ; $P(\bar{A}) = 1 - P(A)$ ; $P(A \cup B) = P(A) + P(B) - P(A \cap B)$. $n!$ rangements, $\binom{n}{k} = \frac{n!}{k!\,(n-k)!}$ choix sans ordre. Indépendance : $P(A \cap B) = P(A)\,P(B)$. $\mathbb{E}[X] = \sum x\,P(X = x)$, $\mathbb{V}(X) = \mathbb{E}[X^2] - \mathbb{E}[X]^2$. Un générateur `np.random.default_rng(graine)`, créé une fois.

## Partie 0 · Vérifier tes exercices papier (✏️ R8.1 à R8.10, 🔗 R8.13)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.5`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"3/11"` ;
- une liste de nombres entre crochets, `[0.25, 0.5, 0.25]` ; vrai ou faux par `True` ou `False`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(3, 11) is accepted too, as "3/11"

**Ex R8.1 — Décrire univers et événements : cartes et pièces**

In [ ]:
# Ex R8.1: the values you found on paper
answer_R8_1a = ...  # a) an integer
answer_R8_1b = ...  # b) an integer
answer_R8_1c = ...  # c) an integer
answer_R8_1d = ...  # d) an integer
answer_R8_1e = ...  # e) an integer
answer_R8_1f = ...  # f) an integer
answer_R8_1g = ...  # g) an integer
answer_R8_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_R8_1a, answer_R8_1b, answer_R8_1c, answer_R8_1d, answer_R8_1e, answer_R8_1f, answer_R8_1g, answer_R8_1h]):
    wbm.check(f"R8.1{letter}", answer)

**Ex R8.2 — Probabilités par équiprobabilité**

In [ ]:
# Ex R8.2: the values you found on paper
answer_R8_2a = ...  # a) an irreducible fraction, as text
answer_R8_2b = ...  # b) an irreducible fraction, as text
answer_R8_2c = ...  # c) an irreducible fraction, as text
answer_R8_2d = ...  # d) an integer
answer_R8_2e = ...  # e) an irreducible fraction, as text
answer_R8_2f = ...  # f) an irreducible fraction, as text
answer_R8_2g = ...  # g) an irreducible fraction, as text
answer_R8_2h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_R8_2a, answer_R8_2b, answer_R8_2c, answer_R8_2d, answer_R8_2e, answer_R8_2f, answer_R8_2g, answer_R8_2h]):
    wbm.check(f"R8.2{letter}", answer)

**Ex R8.3 — P(A ∪ B) et le complémentaire**

In [ ]:
# Ex R8.3: the values you found on paper
answer_R8_3a = ...  # a) a number
answer_R8_3b = ...  # b) a number
answer_R8_3c = ...  # c) a number
answer_R8_3d = ...  # d) a number
answer_R8_3e = ...  # e) True or False
answer_R8_3f = ...  # f) a number
answer_R8_3g = ...  # g) an irreducible fraction, as text
answer_R8_3h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_R8_3a, answer_R8_3b, answer_R8_3c, answer_R8_3d, answer_R8_3e, answer_R8_3f, answer_R8_3g, answer_R8_3h]):
    wbm.check(f"R8.3{letter}", answer)

**Ex R8.4 — Principe multiplicatif et factorielles**

In [ ]:
# Ex R8.4: the values you found on paper
answer_R8_4a = ...  # a) an integer
answer_R8_4b = ...  # b) an integer
answer_R8_4c = ...  # c) an integer
answer_R8_4d = ...  # d) an integer
answer_R8_4e = ...  # e) an integer
answer_R8_4f = ...  # f) an integer
answer_R8_4g = ...  # g) an integer
answer_R8_4h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R8_4a, answer_R8_4b, answer_R8_4c, answer_R8_4d, answer_R8_4e, answer_R8_4f, answer_R8_4g, answer_R8_4h]):
    wbm.check(f"R8.4{letter}", answer)

**Ex R8.5 — Combinaisons et triangle de Pascal**

In [ ]:
# Ex R8.5: the values you found on paper
answer_R8_5a = ...  # a) an integer
answer_R8_5b = ...  # b) an integer
answer_R8_5c = ...  # c) an integer
answer_R8_5d = ...  # d) a list
answer_R8_5e = ...  # e) an integer
answer_R8_5f = ...  # f) an integer
answer_R8_5g = ...  # g) an irreducible fraction, as text
answer_R8_5h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R8_5a, answer_R8_5b, answer_R8_5c, answer_R8_5d, answer_R8_5e, answer_R8_5f, answer_R8_5g, answer_R8_5h]):
    wbm.check(f"R8.5{letter}", answer)

**Ex R8.6 — Arbres pondérés : épreuves successives**

In [ ]:
# Ex R8.6: the values you found on paper
answer_R8_6a = ...  # a) a number
answer_R8_6b = ...  # b) a number
answer_R8_6c = ...  # c) a number
answer_R8_6d = ...  # d) a number
answer_R8_6e = ...  # e) a number
answer_R8_6f = ...  # f) a number
answer_R8_6g = ...  # g) a number
answer_R8_6h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_R8_6a, answer_R8_6b, answer_R8_6c, answer_R8_6d, answer_R8_6e, answer_R8_6f, answer_R8_6g, answer_R8_6h]):
    wbm.check(f"R8.6{letter}", answer)

**Ex R8.7 — Indépendant ou pas ? Le décider par le calcul, avec des cartes et des pièces**

In [ ]:
# Ex R8.7: the values you found on paper
answer_R8_7a = ...  # a) an irreducible fraction, as text
answer_R8_7b = ...  # b) True or False
answer_R8_7c = ...  # c) True or False
answer_R8_7d = ...  # d) True or False
answer_R8_7e = ...  # e) an irreducible fraction, as text
answer_R8_7f = ...  # f) True or False
answer_R8_7g = ...  # g) an irreducible fraction, as text
answer_R8_7h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_R8_7a, answer_R8_7b, answer_R8_7c, answer_R8_7d, answer_R8_7e, answer_R8_7f, answer_R8_7g, answer_R8_7h]):
    wbm.check(f"R8.7{letter}", answer)

**Ex R8.8 — La loi d'une variable aléatoire**

In [ ]:
# Ex R8.8: the values you found on paper
answer_R8_8a = ...  # a) a list (3 decimals)
answer_R8_8b = ...  # b) a number (3 decimals)
answer_R8_8c = ...  # c) a list (4 decimals)
answer_R8_8d = ...  # d) a number (2 decimals)
answer_R8_8e = ...  # e) a number (2 decimals)
answer_R8_8f = ...  # f) a number (2 decimals)
answer_R8_8g = ...  # g) a list (3 decimals)
answer_R8_8h = ...  # h) a number (3 decimals)

for letter, answer in zip("abcdefgh", [answer_R8_8a, answer_R8_8b, answer_R8_8c, answer_R8_8d, answer_R8_8e, answer_R8_8f, answer_R8_8g, answer_R8_8h]):
    wbm.check(f"R8.8{letter}", answer)

**Ex R8.9 — Espérance, variance (avec V = E[X²] − E[X]²) et écart-type**

In [ ]:
# Ex R8.9: the values you found on paper
answer_R8_9a = ...  # a) a number (2 decimals)
answer_R8_9b = ...  # b) a number (2 decimals)
answer_R8_9c = ...  # c) a number (2 decimals)
answer_R8_9d = ...  # d) a number (3 decimals)
answer_R8_9e = ...  # e) a number (2 decimals)
answer_R8_9f = ...  # f) a number (3 decimals)
answer_R8_9g = ...  # g) an irreducible fraction, as text
answer_R8_9h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_R8_9a, answer_R8_9b, answer_R8_9c, answer_R8_9d, answer_R8_9e, answer_R8_9f, answer_R8_9g, answer_R8_9h]):
    wbm.check(f"R8.9{letter}", answer)

**Ex R8.10 — Deux jeux de hasard : équité et risque**

In [ ]:
# Ex R8.10: the values you found on paper
answer_R8_10a = ...  # a) a number
answer_R8_10b = ...  # b) a number
answer_R8_10c = ...  # c) a number
answer_R8_10d = ...  # d) a number
answer_R8_10e = ...  # e) a number
answer_R8_10f = ...  # f) a number
answer_R8_10g = ...  # g) a number
answer_R8_10h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_R8_10a, answer_R8_10b, answer_R8_10c, answer_R8_10d, answer_R8_10e, answer_R8_10f, answer_R8_10g, answer_R8_10h]):
    wbm.check(f"R8.10{letter}", answer)

**Ex R8.13 — Comparer toutes les paires : un tournoi, C(n, 2) et une croissance en n²**

In [ ]:
# Ex R8.13: the values you found on paper
answer_R8_13a = ...  # a) an integer
answer_R8_13b = ...  # b) an integer
answer_R8_13c = ...  # c) an integer
answer_R8_13d = ...  # d) an integer
answer_R8_13e = ...  # e) a number (1 decimal)
answer_R8_13f = ...  # f) an integer

for letter, answer in zip("abcdef", [answer_R8_13a, answer_R8_13b, answer_R8_13c, answer_R8_13d, answer_R8_13e, answer_R8_13f]):
    wbm.check(f"R8.13{letter}", answer)

## Partie A · Ta librairie : dénombrer, énumérer, espérance et variance, simuler

Fiche §2, §3, §7 et §8. Les six fonctions de `mymath/proba_basics.py`.

In [ ]:
# Tools of the notebook (parts A to C)
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex R8.16 — Factorielle et combinaisons · `factorial`, `binomial` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer la factorielle et le coefficient binomial, exactement, avec leurs cas limites.  
**Prérequis :** fiche §3 · **mymath :** `proba_basics.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme dans les chapitres précédents). Ouvre ta copie `mon_travail/mymath/proba_basics.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Comme dans tous les chapitres R0 à R9, Python pur : `fractions.Fraction` est permis (importe-le en haut du fichier : `from fractions import Fraction`, et `import numpy as np` pour `simulate_frequency`), mais ni `math.factorial`, ni `math.comb`, ni `math.perm`, ni le module `statistics`, ni SciPy (la docstring du module le dit, et un test le vérifie) ; seule `simulate_frequency` se sert du générateur de NumPy. Les tests comparent tes résultats à `math.factorial`, `math.comb`, à des comptages exacts avec `Fraction` et à NumPy. Une cellule qui ne s'arrête pas (l'étoile `[*]` reste affichée) : interromps-la (■, *Interrupt*), puis cherche une boucle `while` dont la condition reste toujours vraie.

Écris `factorial(n)` puis `binomial(n, k)` dans `mon_travail/mymath/proba_basics.py` (fiche §3). `factorial` lève une `ValueError` pour un `n` négatif, et `binomial` aussi ; `binomial(n, k)` vaut 0 quand $k < 0$ ou $k > n$, et doit rester un **entier exact**, même pour de grands nombres : pas de division `/`, qui donne un flottant arrondi, mais la division entière `//`.

La vérification calcule :
- a) la liste `[factorial(n) for n in (0, 1, 8, 13)]` ;
- b) le nom de l'erreur levée par `factorial(-2)` (avec `raised`, de la cellule des outils) ;
- c) la ligne $n = 12$ du triangle de Pascal, `[binomial(12, k) for k in range(13)]` ;
- d) la liste `[binomial(9, 11), binomial(9, -1)]` ;
- e) le type de `binomial(60, 30)`, en texte (`"int"` pour un entier).

Puis, dans ta copie : comment calcules-tu `factorial(20) // factorial(18)` de tête ? Pourquoi `binomial(n, k)` gagne-t-il à multiplier et diviser au fur et à mesure, plutôt qu'à calculer trois factorielles, quand $n$ vaut 10 000 et $k$ vaut 3 ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R8")  # reload your saved file

In [ ]:
with checked("R8.16a"):
    wbm.check("R8.16a", [mymath.proba_basics.factorial(n) for n in (0, 1, 8, 13)], computed=True)
with checked("R8.16b"):
    wbm.check("R8.16b", raised(mymath.proba_basics.factorial, -2), computed=True)
with checked("R8.16c"):
    wbm.check("R8.16c", [mymath.proba_basics.binomial(12, k) for k in range(13)], computed=True)
with checked("R8.16d"):
    wbm.check("R8.16d", [mymath.proba_basics.binomial(9, 11), mymath.proba_basics.binomial(9, -1)], computed=True)
with checked("R8.16e"):
    wbm.check("R8.16e", type(mymath.proba_basics.binomial(60, 30)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r8_proba_basics.py", "test_factorial_ or test_binomial_ or test_proba_basics_does")

### Ex R8.17 — Une probabilité exacte par énumération · `enumerate_probability` 🔨 ★★ ⏱️ 25 min
**Objectif :** calculer une probabilité exacte en énumérant des issues équiprobables, avec un événement donné par une fonction.  
**Prérequis :** fiche §2 ; R1 (fractions) ; workbookIA 0A §100.5.3 (fonctions en argument) · **mymath :** `proba_basics.py` · **Parcours :** C

Écris `enumerate_probability(outcomes, event)` : la probabilité exacte de l'événement quand toutes les issues de `outcomes` sont équiprobables, sous la forme d'une `Fraction` (fiche §2). `event` est une **fonction** qui reçoit une issue et renvoie `True` si elle est favorable (workbookIA 0A §100.5.3). Attention : `outcomes` peut être une liste, mais aussi un **générateur**, qui n'a pas de `len()` et ne se lit qu'une fois ; compte donc les issues en les parcourant. Sans aucune issue, lève une `ValueError`.

La cellule fournie définit `three_d4`, les 64 triplets de trois lancers d'un dé à 4 faces. La vérification calcule :
- a) la probabilité que la somme des trois dés vaille 6 ;
- b) la probabilité qu'au moins deux des trois dés montrent le même numéro ;
- c) avec le générateur `(n for n in range(1, 1001))`, la probabilité qu'un entier de 1 à 1 000 tiré au hasard s'écrive avec au moins un chiffre 7 ;
- d) le nom de l'erreur levée pour une liste d'issues vide ;
- e) le type du résultat de a), en texte.

Puis, dans ta copie : retrouve a) en dénombrant à la main les triplets de somme 6, et b) par le contraire (« trois numéros différents »).

In [ ]:
three_d4 = [(a, b, c) for a in range(1, 5) for b in range(1, 5) for c in range(1, 5)]   # 64 triplets

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R8")  # reload your saved file

In [ ]:
with checked("R8.17a"):
    wbm.check("R8.17a", mymath.proba_basics.enumerate_probability(three_d4, lambda t: sum(t) == 6), computed=True)
with checked("R8.17b"):
    wbm.check("R8.17b", mymath.proba_basics.enumerate_probability(three_d4, lambda t: len(set(t)) < 3),
              computed=True)
with checked("R8.17c"):
    wbm.check("R8.17c", mymath.proba_basics.enumerate_probability((n for n in range(1, 1001)),
                                                                 lambda n: "7" in str(n)), computed=True)
with checked("R8.17d"):
    wbm.check("R8.17d", raised(mymath.proba_basics.enumerate_probability, [], lambda o: True), computed=True)
with checked("R8.17e"):
    result = mymath.proba_basics.enumerate_probability(three_d4, lambda t: sum(t) == 6)
    wbm.check("R8.17e", type(result).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r8_proba_basics.py", "test_enumerate_probability_ or test_proba_basics_does")

### Ex R8.18 — Espérance et variance d'une loi · `expectation`, `variance` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer l'espérance et la variance d'une loi finie, exactes avec des fractions, et vérifier la loi d'abord.  
**Prérequis :** fiche §7 · **mymath :** `proba_basics.py` · **Parcours :** C, E

Écris `expectation(values, probabilities)` puis `variance(values, probabilities)` (fiche §7). Les deux commencent par **vérifier la loi** : autant de valeurs que de probabilités, aucune probabilité négative, une somme égale à 1 (à $10^{-9}$ près, pour les flottants) ; sinon, une `ValueError`. Le résultat est une `Fraction` quand toutes les valeurs et toutes les probabilités sont des entiers ou des fractions, un flottant sinon. Pour la variance, pars de la définition $\mathbb{E}\big[(X - \mathbb{E}[X])^2\big]$, en réutilisant `expectation`.

La cellule fournie définit la loi du nombre de 6 en deux lancers d'un dé (`sixes_values`, `sixes_probs`, en fractions) et une loi en flottants (`float_values`, `float_probs`). La vérification calcule :
- a) et b) l'espérance et la variance de la loi du nombre de 6 ;
- c) et d) l'espérance et la variance de la loi en flottants ;
- e) le nom de l'erreur levée par `expectation([1, 2, 3], [0.5, 0.5])`, puis f) par `variance([1, 2], [0.7, 0.2])` ;
- g) le type du résultat de b), en texte.

Puis, dans ta copie : retrouve a) et b) à la main, avec la formule $\mathbb{V}(X) = \mathbb{E}[X^2] - \mathbb{E}[X]^2$.

In [ ]:
sixes_values = [0, 1, 2]                                          # number of 6 in two rolls of a die
sixes_probs = [Fraction(25, 36), Fraction(10, 36), Fraction(1, 36)]
float_values = [-1.5, 0, 2.5]
float_probs = [0.3, 0.45, 0.25]

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R8")  # reload your saved file

In [ ]:
with checked("R8.18a"):
    wbm.check("R8.18a", mymath.proba_basics.expectation(sixes_values, sixes_probs), computed=True)
with checked("R8.18b"):
    wbm.check("R8.18b", mymath.proba_basics.variance(sixes_values, sixes_probs), computed=True)
with checked("R8.18c"):
    wbm.check("R8.18c", mymath.proba_basics.expectation(float_values, float_probs), computed=True)
with checked("R8.18d"):
    wbm.check("R8.18d", mymath.proba_basics.variance(float_values, float_probs), computed=True)
with checked("R8.18e"):
    wbm.check("R8.18e", raised(mymath.proba_basics.expectation, [1, 2, 3], [0.5, 0.5]), computed=True)
with checked("R8.18f"):
    wbm.check("R8.18f", raised(mymath.proba_basics.variance, [1, 2], [0.7, 0.2]), computed=True)
with checked("R8.18g"):
    wbm.check("R8.18g", type(mymath.proba_basics.variance(sixes_values, sixes_probs)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r8_proba_basics.py", "test_expectation_ or test_variance_ or test_proba_basics_does")

### Ex R8.19 — Estimer une probabilité par simulation · `simulate_frequency` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer une simulation reproductible : un générateur créé une fois, une expérience répétée, une fréquence.  
**Prérequis :** fiche §8 ; workbookIA 0A §100.8.7 (default_rng) · **mymath :** `proba_basics.py` · **Parcours :** C, E

Écris `simulate_frequency(experiment, n_trials, seed=0)` (fiche §8) : crée **un** générateur `np.random.default_rng(seed)`, appelle `experiment(rng)` `n_trials` fois avec lui, et renvoie la fréquence des succès (le nombre de `True` divisé par `n_trials`), en flottant. Un `n_trials` nul ou négatif lève une `ValueError`. `experiment` est une fonction qui fait **un** essai avec le générateur qu'elle reçoit, par exemple `lambda rng: rng.integers(1, 5) == 4` (« le dé à 4 faces donne 4 »).

La vérification calcule :
- a) la fréquence de « le dé à 4 faces donne 4 » sur 1 000 essais, avec la graine 5 ;
- b) si deux appels avec la même graine donnent la même fréquence (vrai ou faux) ;
- c) si les graines 5 et 6 donnent des fréquences différentes (vrai ou faux) ;
- d) le nom de l'erreur levée pour 0 essai ;
- e) si la fréquence de « la somme de deux dés à 4 faces vaut au moins 6 », sur 20 000 essais avec la graine 0, est à moins de 0,02 de la probabilité $\frac{3}{8}$ du §6 (vrai ou faux).

Puis, dans ta copie : pourquoi la fréquence de a) n'est-elle pas exactement $\frac{1}{4}$ ? Que se passerait-il si tu créais le générateur **dans** la boucle, avec la même graine à chaque essai ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R8")  # reload your saved file

In [ ]:
one_four = lambda rng: rng.integers(1, 5) == 4                                  # noqa: E731
with checked("R8.19a"):
    wbm.check("R8.19a", mymath.proba_basics.simulate_frequency(one_four, 1000, seed=5), computed=True)
with checked("R8.19b"):
    first = mymath.proba_basics.simulate_frequency(one_four, 1000, seed=5)
    second = mymath.proba_basics.simulate_frequency(one_four, 1000, seed=5)
    wbm.check("R8.19b", bool(first == second), computed=True)
with checked("R8.19c"):
    other = mymath.proba_basics.simulate_frequency(one_four, 1000, seed=6)
    wbm.check("R8.19c", bool(first != other), computed=True)
with checked("R8.19d"):
    wbm.check("R8.19d", raised(mymath.proba_basics.simulate_frequency, one_four, 0), computed=True)
with checked("R8.19e"):
    sum_at_least_6 = lambda rng: rng.integers(1, 5) + rng.integers(1, 5) >= 6       # noqa: E731
    frequency = mymath.proba_basics.simulate_frequency(sum_at_least_6, 20_000, seed=0)
    wbm.check("R8.19e", bool(abs(frequency - 3 / 8) < 0.02), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r8_proba_basics.py", "test_simulate_frequency_ or test_proba_basics_does")

## Partie B · Une prédiction, la loi des grands nombres et une figure

Fiche §2 à §8. Une probabilité qui surprend presque tout le monde, des fréquences qui se resserrent autour d'une probabilité, puis une loi en image.

### Ex R8.20 — Prédire : dans une classe de 30, deux élèves ont-ils le même anniversaire ? 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir une probabilité contre-intuitive, la vérifier par simulation, puis la calculer exactement.  
**Prérequis :** fiche §2, §3 et §8 · **Parcours :** C, E

On suppose que les 365 jours de l'année (sans 29 février) sont équiprobables pour une date d'anniversaire, et que les élèves d'une classe sont nés indépendamment les uns des autres. **Avant d'exécuter quoi que ce soit**, prédis :
- a) dans une classe de 30 élèves, la probabilité qu'au moins deux élèves aient le même anniversaire : (1) moins de 10 % ; (2) entre 10 % et 50 % ; (3) plus de 50 % ;
- b) le plus petit nombre d'élèves pour lequel cette probabilité dépasse 50 % est proche de : (1) 23 ; (2) 60 ; (3) 183 ;
- c) avec 60 élèves, la probabilité est : (1) environ 70 % ; (2) environ 85 % ; (3) plus de 99 %.

Réponds par le numéro du choix : 1, 2 ou 3.

Écris tes trois prédictions, vérifie-les, puis lance l'expérience, qui simule 20 000 classes de chaque taille. Ensuite, calcule la probabilité **exacte** :
- d) écris `p_shared(n)`, la probabilité qu'au moins deux élèves sur $n$ aient le même anniversaire. Passe par le contraire, « tous les anniversaires sont différents » : le premier élève a 365 jours possibles sur 365, le deuxième 364 sur 365, le troisième 363 sur 365… (une boucle, ou un produit Π). La vérification calcule `p_shared(30)` (3 décimales) ;
- e) écris `smallest_class()`, qui renvoie le plus petit nombre d'élèves $n$ pour lequel `p_shared(n)` dépasse 0,5 (un entier, cherché avec une boucle `for n in range(1, 366)`).

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R8_20a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_R8_20b = ...  # b) 1, 2 or 3
prediction_R8_20c = ...  # c) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abc", [prediction_R8_20a, prediction_R8_20b, prediction_R8_20c]):
    wbm.check(f"R8.20{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes trois prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R8_20a, prediction_R8_20b, prediction_R8_20c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    rng = np.random.default_rng(2026)
    for n_students in (10, 23, 30, 60):
        classes = rng.integers(0, 365, size=(20_000, n_students))     # 20 000 classes, one birthday per student
        shared = np.any(np.diff(np.sort(classes, axis=1), axis=1) == 0, axis=1)
        print(f"{n_students:>2} élèves : fréquence d'un anniversaire partagé = {shared.mean():.3f}")

**d) et e)** La probabilité exacte, par le contraire.

In [ ]:
def p_shared(n):
    """The probability that at least two of n students share a birthday (365 equally likely days)."""
    raise NotImplementedError("passe par le contraire : le produit des (365 - k) / 365")


def smallest_class():
    """The smallest number of students n for which p_shared(n) > 0.5 (a loop: for n in range(1, 366))."""
    raise NotImplementedError("une boucle for sur n, qui s'arrête au premier p_shared(n) > 0.5")

In [ ]:
with checked("R8.20d"):
    wbm.check("R8.20d", p_shared(30), computed=True)
with checked("R8.20e"):
    wbm.check("R8.20e", smallest_class(), computed=True)

**Après l'expérience**, dans ta copie : combien de **paires** d'élèves une classe de 30 contient-elle (§3) ? Pourquoi ce nombre explique-t-il que la probabilité soit si grande, alors que chaque paire n'a qu'une chance sur 365 de partager un anniversaire ?

### Ex R8.21 — La loi des grands nombres : fréquence et nombre d'essais 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer comment la part des séries dont la fréquence s'écarte de la probabilité diminue quand le nombre d'essais grandit.  
**Prérequis :** fiche §8 ; workbookIA 0A §100.8.7 (default_rng) et §100.8.3 (calcul vectorisé) · **Parcours :** C, E

Une roue a 3 secteurs égaux, numérotés 0, 1 et 2, et l'on s'intéresse à l'événement « la roue s'arrête sur le 0 », de probabilité $\frac{1}{3}$. Pour chaque nombre de tours $n$ de `sizes = [10, 100, 1000, 10000]`, **dans cet ordre**, et avec **un seul** générateur `rng = np.random.default_rng(81)`, créé avant la boucle :
- tire 500 séries de $n$ tours : `freqs = (rng.integers(0, 3, size=(500, n)) == 0).mean(axis=1)`, les 500 fréquences de l'événement ;
- calcule la proportion des 500 séries dont la fréquence s'écarte de $\frac{1}{3}$ de plus de 0,05 : `np.mean(np.abs(freqs - 1 / 3) > 0.05)`.

Questions :
- a) `far` : la liste des quatre proportions, dans l'ordre des $n$ (3 décimales) ;
- b) `mean_10000` : pour $n = 10\,000$, la moyenne des 500 fréquences (4 décimales) ;
- c) vers quel nombre la proportion de a) semble-t-elle tendre quand $n$ grandit ?

Trace aussi, côte à côte, l'histogramme des 500 fréquences pour $n = 100$ et pour $n = 10\,000$, sur le même axe horizontal de 0 à 1 (garde ces fréquences dans un dictionnaire pendant la boucle).

Dans ta copie : que dit la loi des grands nombres, et que ne dit-elle pas ? Avec $n = 10\,000$, que deviendrait la proportion de a) pour un écart de 0,01 au lieu de 0,05 ?

In [ ]:
sizes = [10, 100, 1000, 10000]

In [ ]:
far = ...           # a) a list of 4 proportions
mean_10000 = ...    # b) a number
answer_R8_21c = ... # c) a number

In [ ]:
wbm.check("R8.21a", far)
wbm.check("R8.21b", mean_10000)
wbm.check("R8.21c", answer_R8_21c)

### Ex R8.22 — Le diagramme en bâtons d'une loi 🎨 ★ ⏱️ 10 min
**Objectif :** tracer la loi d'une variable aléatoire discrète et y placer son espérance.  
**Prérequis :** fiche §6 et §7 ; R0 (matplotlib) · **Parcours :** C

Complète `plot_law(ax, values, probabilities)`, qui dessine dans l'axe `ax` qu'elle reçoit :
- un bâton par valeur, centré sur la valeur, de hauteur sa probabilité, avec `ax.bar(values, probabilities, width=0.2)` (la vérification lit les rectangles de `ax.bar`) ;
- une droite verticale en pointillés à l'espérance, **calculée dans la fonction** à partir de `values` et `probabilities` (`ax.axvline`) ;
- un titre, et le nom des deux axes.

La cellule suivante trace la loi du nombre de piles en quatre lancers d'une pièce (`piles_values`, `piles_probs`, fournies), puis une autre la vérifie, sur cette loi et sur une loi qui n'est pas symétrique.

Dans ta copie : où se trouve l'espérance, et pourquoi pouvait-on la prévoir sans calcul ? Pourquoi la loi n'est-elle pas « plate », alors que les 16 suites de quatre lancers sont équiprobables ?

In [ ]:
piles_values = [0, 1, 2, 3, 4]                       # number of piles in four coin flips
piles_probs = [1 / 16, 4 / 16, 6 / 16, 4 / 16, 1 / 16]

In [ ]:
def plot_law(ax, values, probabilities):
    """Draw the law (one bar per value) in ax, with a dashed vertical line at the expectation."""
    raise NotImplementedError("ax.bar, puis ax.axvline à l'espérance, un titre et les axes")

In [ ]:
with checked("R8.22 figure"):
    fig, ax = plt.subplots(figsize=(6, 3.5))
    try:
        created = drawn(plot_law, ax, piles_values, piles_probs)
    except BaseException:
        plt.close(fig)                                     # no empty figure while the function is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex R8.22 : ta fonction ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans l'axe "
              "qu'elle reçoit.")
    else:
        plt.show()

**Vérifications automatiques** (la figure elle-même se juge à l'œil).

In [ ]:
with checked("R8.22 vérifications"):
    problems = []
    for values, probs in [(piles_values, piles_probs), ([0, 1, 5], [0.5, 0.3, 0.2])]:   # symmetric, then not
        fig, ax = plt.subplots(figsize=(6, 3.5))
        try:
            created = drawn(plot_law, ax, values, probs)
        finally:
            plt.close(fig)                                 # only the checks below are shown here
        if created:
            problems.append("ta fonction ouvre sa propre figure : dessine dans l'axe reçu (ax.bar, …)")
            break
        heights = sorted((round(float(p.get_x() + p.get_width() / 2), 6), round(float(p.get_height()), 6))
                         for p in ax.patches)
        expected = sorted((float(v), round(float(p), 6)) for v, p in zip(values, probs))
        if heights != expected:
            problems.append("un bâton par valeur, centré sur la valeur, de hauteur sa probabilité (ax.bar)")
        mean = sum(v * p for v, p in zip(values, probs))
        verticals = [l for l in ax.lines if len(set(np.round(np.asarray(l.get_xdata(), dtype=float), 6))) == 1]
        at_mean = [l for l in verticals if np.isclose(float(np.asarray(l.get_xdata(), dtype=float)[0]), mean)]
        if not at_mean:
            problems.append("une droite verticale à l'espérance (ax.axvline), calculée dans la fonction : la moyenne "
                            "des valeurs pondérée par leurs probabilités (vérifiée aussi sur une loi non symétrique)")
        elif not any(l.get_linestyle() == "--" for l in at_mean):
            problems.append("la droite de l'espérance en pointillés (linestyle=\"--\")")
        if not ax.get_title() or not ax.get_xlabel() or not ax.get_ylabel():
            problems.append("un titre et le nom des deux axes (ax.set_title, ax.set_xlabel, ax.set_ylabel)")
        if problems:
            break
    if problems:
        print("❌ Ex R8.22 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R8.22 : les bâtons, l'espérance (sur deux lois), le titre et les axes.")

## Partie C · Raisonnements faux et reproductibilité

Fiche §2, §5 et §8. Trois raisonnements faux à trouver, puis une expérience aléatoire reproductible.

### Ex R8.23 — Chasse à l'erreur : trois raisonnements de probabilités faux 🐛 ★★ ⏱️ 20 min
**Objectif :** repérer des raisonnements faux (réunion, équiprobabilité, indépendance), puis calculer les bonnes probabilités.  
**Prérequis :** fiche §2 et §5 · **Parcours :** C, E

Voici cinq raisonnements. Trois sont faux. **Sans calculer d'abord**, trouve-les.

- **(A)** « On lance deux fois un dé à 4 faces. La probabilité d'obtenir au moins un 4 est $\frac{1}{4} + \frac{1}{4} = \frac{1}{2}$. »
- **(B)** « On lance deux pièces. On obtient 0, 1 ou 2 piles : trois cas, donc $P(\text{exactement un pile}) = \frac{1}{3}$. »
- **(C)** « $P(A) = 0{,}5$, $P(B) = 0{,}4$, et $A$ et $B$ sont indépendants, donc $P(A \cap B) = 0{,}5 \times 0{,}4 = 0{,}2$. »
- **(D)** « Une roue a 10 secteurs égaux, dont 3 rouges et 2 verts. Un secteur n'est jamais rouge et vert à la fois, donc $P(\text{rouge ou vert}) = \frac{3}{10} + \frac{2}{10} = \frac{1}{2}$. »
- **(E)** « $A$ et $B$ sont incompatibles, avec $P(A) = 0{,}3$ et $P(B) = 0{,}5$. Ils sont donc indépendants, et $P(A \cap B) = 0{,}3 \times 0{,}5 = 0{,}15$. »

Questions :
- a) les lettres des raisonnements faux, dans l'ordre alphabétique (un texte) ;
- b) les **bonnes** valeurs des probabilités que calculent les trois raisonnements faux, dans l'ordre alphabétique de leurs lettres (une liste, 4 décimales).

Dans ta copie : pour chaque raisonnement faux, quelle règle a été oubliée ? Lequel des deux raisonnements justes pouvait sembler faux, et pourquoi ne l'est-il pas ?

In [ ]:
answer_R8_23a = ...  # a) letters, as text
answer_R8_23b = ...  # b) a list of 3 probabilities (4 decimals)

In [ ]:
wbm.check("R8.23a", answer_R8_23a)
wbm.check("R8.23b", answer_R8_23b)

### Ex R8.24 — Reproductibilité : générateur aléatoire et graine 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire une expérience aléatoire reproductible, et reconnaître le bug du générateur recréé à chaque tour.  
**Prérequis :** fiche §8 ; workbookIA 0A §100.8.7 · **Parcours :** C, E

Dans workbookIA, chaque expérience fixe sa graine, pour que tes résultats soient ceux du corrigé. La fonction `measure(rng)` (fournie) simule un score bruité : l'accuracy d'un modèle mesurée sur 200 exemples de test tirés avec le générateur `rng`.

1. Écris `run_experiments(n_runs, seed)`, qui renvoie la liste de `n_runs` mesures, avec **un seul** générateur `np.random.default_rng(seed)`, créé avant la boucle.
2. La cellule de vérification contrôle que la même graine redonne les mêmes mesures, que les mesures diffèrent entre elles, et qu'une autre graine donne d'autres mesures ; puis elle affiche le résultat de `buggy_runs(5, 42)` (fournie).

Dans ta copie : que remarques-tu sur les mesures de `buggy_runs` ? Quelle ligne cause ce bug, et pourquoi ? Que mesure l'écart entre tes 5 mesures, et pourquoi ne faut-il pas comparer deux modèles sur une seule mesure ?

In [ ]:
def measure(rng):
    """A noisy score: the accuracy of a model measured on 200 test examples drawn with rng."""
    return float((rng.random(200) < 0.85).mean())


def buggy_runs(n_runs, seed):
    """n_runs measures... with a classic bug."""
    results = []
    for _ in range(n_runs):
        rng = np.random.default_rng(seed)
        results.append(measure(rng))
    return results

In [ ]:
def run_experiments(n_runs, seed):
    """n_runs measures, with ONE generator default_rng(seed) created before the loop."""
    raise NotImplementedError("un générateur, puis une boucle qui appelle measure(rng)")

In [ ]:
with checked("R8.24 vérifications"):
    runs = run_experiments(5, seed=42)
    problems = []
    if runs is None:
        problems.append("run_experiments renvoie None : as-tu oublié le return de la liste ?")
    elif not isinstance(runs, list) or len(runs) != 5:
        problems.append("run_experiments(5, seed) renvoie une liste de 5 mesures")
    else:
        if run_experiments(5, seed=42) != runs:
            problems.append("la même graine doit redonner exactement les mêmes mesures")
        if len(set(runs)) < 5:
            problems.append("les 5 mesures doivent différer : un seul générateur, créé avant la boucle, qui avance")
        if run_experiments(5, seed=43) == runs:
            problems.append("une autre graine doit donner d'autres mesures")
    if problems:
        print("❌ Ex R8.24 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R8.24 : reproductible avec la même graine, différent d'une mesure à l'autre et d'une graine à "
              "l'autre.")
    print("buggy_runs(5, 42) :", buggy_runs(5, 42))

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Calcules-tu une probabilité par équiprobabilité, par le contraire ou avec un arbre ?
2. Sais-tu choisir entre $n^k$, les arrangements, $n!$ et $\binom{n}{k}$, et tester une indépendance ?
3. Calcules-tu l'espérance et la variance d'une loi, et sais-tu dire ce qu'affirme la loi des grands nombres ?

**La suite** : R9, *Géométrie du plan, vecteurs et trigonométrie*. Dans workbookIA, continue le chapitre 0A au rythme de ton tableau de bord.